In [1]:
import pandas as pd
import numpy as np

# load dataset 

normal_dataset_path = '../dataset/clean/normal/normal.csv'

attack_0rtt_dataset_path = '../dataset/clean/tls/attack_0rtt_dataset.csv'
attack_heartbleed_dataset_path = '../dataset/clean/tls/attack_heartbleed_dataset.csv'


attack_cert_probe_dataset_path = '../dataset/clean/probe/attack_cert_probe_dataset.csv'
attack_crypto_probe_dataset_path = '../dataset/clean/probe/attack_crypto_probe_dataset.csv'
attack_cve_probe_dataset_path = '../dataset/clean/probe/attack_cve_probe_dataset.csv'
attack_protocol_probe_dataset_path = '../dataset/clean/probe/attack_protocol_probe_dataset.csv'

attack_goldeneye_dataset_path = '../dataset/clean/dos/attack_goldeneye_dataset.csv'
attack_hulk_dataset_path = '../dataset/clean/dos/attack_hulk_dataset.csv'
attack_rst_flood_dataset_path = '../dataset/clean/dos/attack_rst_flood_dataset.csv'
attack_slowloris_dataset_path = '../dataset/clean/dos/attack_slowloris_dataset.csv'
attack_sync_flood_dataset_path = '../dataset/clean/dos/attack_sync_flood_dataset.csv'
attack_tcp_ack_dataset_path = '../dataset/clean/dos/attack_tcp_ack_dataset.csv'
attack_tors_dataset_path = '../dataset/clean/dos/attack_tors_dataset.csv'
attack_udp_dataset_path = '../dataset/clean/dos/attack_udp_dataset.csv'




normal_dataset = pd.read_csv(normal_dataset_path)

attack_0rtt_dataset = pd.read_csv(attack_0rtt_dataset_path)
attack_cert_probe_dataset = pd.read_csv(attack_cert_probe_dataset_path)
attack_crypto_probe_dataset = pd.read_csv(attack_crypto_probe_dataset_path)
attack_cve_probe_dataset = pd.read_csv(attack_cve_probe_dataset_path)
attack_goldeneye_dataset = pd.read_csv(attack_goldeneye_dataset_path)
attack_heartbleed_dataset = pd.read_csv(attack_heartbleed_dataset_path)
attack_hulk_dataset = pd.read_csv(attack_hulk_dataset_path)
attack_protocol_probe_dataset = pd.read_csv(attack_protocol_probe_dataset_path)
attack_rst_flood_dataset = pd.read_csv(attack_rst_flood_dataset_path)
attack_slowloris_dataset = pd.read_csv(attack_slowloris_dataset_path)
attack_sync_flood_dataset = pd.read_csv(attack_sync_flood_dataset_path)
attack_tcp_ack_dataset = pd.read_csv(attack_tcp_ack_dataset_path)
attack_tors_dataset = pd.read_csv(attack_tors_dataset_path)
attack_udp_dataset = pd.read_csv(attack_udp_dataset_path)


# Randomly sample data

# Normal
normal_dataset = normal_dataset.sample(n=49000, random_state=42) 

# TLS (use full)
tls_dataset = pd.concat(
    [
        attack_0rtt_dataset,
        attack_heartbleed_dataset
    ],
    axis=0,
    ignore_index=True
)

# Probe 
probe_dataset = pd.concat(
    [
        attack_cert_probe_dataset.sample(n=1000, random_state=42),
        attack_crypto_probe_dataset.sample(n=1000, random_state=42) ,
        attack_cve_probe_dataset.sample(n=2000, random_state=42) ,
        attack_protocol_probe_dataset.sample(n=1000, random_state=42) 
    ],
    axis=0,
    ignore_index=True
)

# Dos
dos_dataset = pd.concat(
    [
        attack_goldeneye_dataset.sample(n=1000, random_state=42),
        attack_hulk_dataset.sample(n=1000, random_state=42),
        attack_rst_flood_dataset,
        attack_slowloris_dataset,
        attack_sync_flood_dataset.sample(n=1000, random_state=42),
        attack_tcp_ack_dataset.sample(n=1000, random_state=42),
        attack_tors_dataset.sample(n=1000, random_state=42),
        attack_udp_dataset
    ],
    axis=0,
    ignore_index=True
)



In [2]:
normal_df = normal_dataset
attack_df = pd.concat([tls_dataset, probe_dataset, dos_dataset], ignore_index=True)

# Combine all
dataset_df = pd.concat([normal_df, attack_df], ignore_index=True)

#  Shuffle the data
dataset_df = dataset_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Result
print("Combined shape:", dataset_df.shape)
print(dataset_df['label'].value_counts())

Combined shape: (59380, 77)
label
normal    49000
ddos       5264
probe      5000
tls         116
Name: count, dtype: int64


# IDS CNN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score

# ===== 1) Data =====
X = dataset_df.drop(columns=['label']).values          # shape: (N, 77)
y_str = dataset_df['label'].values

le = LabelEncoder()
y = le.fit_transform(y_str)
classes = le.classes_
num_classes = len(classes)
print("Classes:", classes)

# Scaling helps CNN on tabular data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# Train/val/test: 70/10/20
X_tr_full, X_te, y_tr_full, y_te = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr_full, y_tr_full, test_size=0.125, random_state=42, stratify=y_tr_full
)  # 0.125 of 0.8 ≈ 0.10

# Reshape to (N, C=1, L=77) for Conv1d
def to_tensor3(x):
    return torch.tensor(x, dtype=torch.float32).unsqueeze(1)  # add channel dim

X_tr_t  = to_tensor3(X_tr)
X_val_t = to_tensor3(X_val)
X_te_t  = to_tensor3(X_te)

y_tr_t  = torch.tensor(y_tr, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_te_t  = torch.tensor(y_te, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t),  batch_size=64, shuffle=False)

# ===== 2) Class weights (to help the tiny 'tls' class) =====
unique, counts = np.unique(y_tr, return_counts=True)
freq = dict(zip(unique, counts))
inv = {k: 1.0/v for k, v in freq.items()}
scale = np.mean(list(inv.values()))
cls_weights_np = np.array([inv[k]/scale for k in sorted(inv.keys())], dtype=np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cls_weights = torch.tensor(cls_weights_np, dtype=torch.float32, device=device)

# ===== 3) Model: 1D CNN over features =====
class CNN1D(nn.Module):
    def __init__(self, in_channels=1, num_classes=4, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            # Block 1
            nn.Conv1d(in_channels, 64, kernel_size=5, stride=1, padding=2),  # "same" length
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Block 2
            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Block 3
            nn.Conv1d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            # Global average pooling to (B, C, 1)
            nn.AdaptiveAvgPool1d(1)
        )
        self.head = nn.Linear(128, num_classes)

    def forward(self, x):             # x: (B, 1, 77)
        z = self.net(x)               # (B, 128, 1)
        z = z.squeeze(-1)             # (B, 128)
        return self.head(z)           # logits

model = CNN1D(in_channels=1, num_classes=num_classes, dropout=0.3).to(device)

# ===== 4) Loss & Optimizer =====
criterion = nn.CrossEntropyLoss(weight=cls_weights)  # class-weighted CE
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)  # L2 wd

# ===== 5) Train / Validate =====
epochs = 50
best_val_acc = 0.0

for epoch in range(1, epochs + 1):
    # Train
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    correct, total, val_loss = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    val_acc = correct / total
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_cnn1d.pth")

    print(f"Epoch [{epoch}/{epochs}] "
          f"TrainLoss: {train_loss/len(train_loader):.4f}  "
          f"ValLoss: {val_loss/len(val_loader):.4f}  "
          f"ValAcc: {val_acc:.4f}")
    

# ===== 6) Test =====
model.load_state_dict(torch.load("best_cnn1d.pth", map_location=device))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        y_true.extend(yb.numpy())
        y_pred.extend(preds)

print("TEST accuracy:", accuracy_score(y_true, y_pred))
print("TEST balanced_accuracy:", balanced_accuracy_score(y_true, y_pred))
print("TEST macro F1:", f1_score(y_true, y_pred, average="macro"))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=classes))

Classes: ['ddos' 'normal' 'probe' 'tls']
Epoch [1/50] TrainLoss: 0.5748  ValLoss: 0.4137  ValAcc: 0.9589
Epoch [2/50] TrainLoss: 0.3919  ValLoss: 0.3223  ValAcc: 0.9672
Epoch [3/50] TrainLoss: 0.3316  ValLoss: 0.2839  ValAcc: 0.9695
Epoch [4/50] TrainLoss: 0.2866  ValLoss: 0.2511  ValAcc: 0.9697
Epoch [5/50] TrainLoss: 0.2709  ValLoss: 0.2211  ValAcc: 0.9702
Epoch [6/50] TrainLoss: 0.2312  ValLoss: 0.1903  ValAcc: 0.9678
Epoch [7/50] TrainLoss: 0.2266  ValLoss: 0.2236  ValAcc: 0.9720
Epoch [8/50] TrainLoss: 0.2225  ValLoss: 0.1851  ValAcc: 0.9739
Epoch [9/50] TrainLoss: 0.2013  ValLoss: 0.1963  ValAcc: 0.9788
Epoch [10/50] TrainLoss: 0.1845  ValLoss: 0.1893  ValAcc: 0.9870
Epoch [11/50] TrainLoss: 0.1613  ValLoss: 0.1773  ValAcc: 0.9655
Epoch [12/50] TrainLoss: 0.1786  ValLoss: 0.1646  ValAcc: 0.9776
Epoch [13/50] TrainLoss: 0.1746  ValLoss: 0.1544  ValAcc: 0.9717
Epoch [14/50] TrainLoss: 0.1680  ValLoss: 0.1544  ValAcc: 0.9756
Epoch [15/50] TrainLoss: 0.1402  ValLoss: 0.1455  ValAcc: 

/tmp/ipykernel_2879683/1856934183.py:137: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_cnn1d.pth", map_location=device))
